# Setup

In [1]:
import sys
sys.path.append('../../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
from pandas import DataFrame
from processor.llm.interface.model_interface import ModelInterface
from processor.llm.interface.model_factory import get_model
from processor.utils import format_schema_with_samples
from tqdm import tqdm
from processor.types.message import Message

In [2]:
ckp = '../llm/weight/qwen25-7b'
interface = get_model(ckp)
model: ModelInterface = interface(ckp)
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# Step 1: Schema Enhancer

In [28]:
schema_enhancer_prompts = {
    "table_descriptor": "You are an experienced data scientist. You are given the schema of a table, along with some sample row(s), with the pipe character (`|`) as the separators of columns and row values. Your goal is to briefly guess what the table likely represents. Output your guess directly without any extra formatting.",
    "column_renamer": 'You are an experienced data scientist. You are given:\n\n- The schema of a table, along with some sample row(s), with the pipe character (`|`) as the separators of columns and row values.\n- A description of what the table represents.\n- A column from the schema to be renamed.\n\nYour goal is to rename the specified column to make it more explicit and descriptive while considering the other columns in the schema and the overall description of the table. Start by thinking for a bit and end your thought with this exact format (to ease parsing of your answer):\n\nNew column name: ...',
}

In [29]:
def __get_table_description(table: DataFrame, sample = 3):
        table_desc_samples: list[str] = []
        for i in range(sample):
            msg: list[Message] = [
                {'role': 'system', 'content': schema_enhancer_prompts['table_descriptor']},
                {'role': 'user', 'content': format_schema_with_samples(table, 3, 42+i) },
            ]
            table_desc_sample = model.chat(msg)
            table_desc_samples.append(table_desc_sample)

        # Now combine them
        all_descs = ""
        for j in range(len(table_desc_samples)):
            all_descs += f'Description {j}: {table_desc_samples[j]}\n'
        all_descs = all_descs.strip()
        msg: list[Message] = [
            {'role': 'system', 'content': 'You are an expert data scientist skilled in precise reasoning. Your task is to synthesize a single, concise description from several similar ones. Always choose the most specific term when multiple levels of abstraction are mentioned (e.g., if both "retail stores" and "businesses" are mentioned, only use "retail stores"). Do not include both general and specific terms together. Be concise, avoid repetition, and return only the refined description—no extra formatting or commentary.'},
            {'role': 'user', 'content': all_descs },
        ]
        table_description = model.chat(msg)
        print(f"==> Sample descriptions: {table_desc_samples}")
        print(f"==> Table description: {table_description}")
        return table_description

def __rename_column(table: DataFrame, col_name: str, table_desc: str):
    msg: list[Message] = [
        {'role': 'system', 'content': schema_enhancer_prompts['column_renamer']},
        {'role': 'user', 'content': f'- Schema: {format_schema_with_samples(table)}\n\n- Description: {table_desc}\n\n- Column to be renamed: {col_name}' },
    ]
    new_col_name = model.chat(msg)
    return new_col_name

In [30]:
def get_enhanced_schema(tables: list[DataFrame]) -> list[str]:
        results = []
        for table in tqdm(tables):
            # Step 1: Determine what the table represents
            table_description = __get_table_description(table)
            print(f'=> Schema description: {table_description}')

            # Step 2: Rename each column
            new_columns: list[str] = []
            print(f"=> Overall schema: {table.columns}")
            for col in table.columns:
                new_col_reason_and_name = __rename_column(table, col, table_description)
                new_col_name = new_col_reason_and_name.split('New column name:')[-1].strip()
                print(f"==> Reasoning: {new_col_reason_and_name}")
                print(f"==> Renaming column {col} to {new_col_name}")
                new_columns.append(new_col_name)
            results.append(new_columns)
        return results

In [31]:
df = pd.read_csv('../../../data_src/zomato.csv')
get_enhanced_schema([df])

  0%|          | 0/1 [00:00<?, ?it/s]

==> Sample descriptions: ['This table likely represents restaurant or business listings, including their ratings, contact information, and addresses.', 'The table likely represents a list of restaurants or businesses along with their ratings, contact information, and address.', 'The table likely represents information about restaurants, including their name, rating, contact number, number of reviews, and address.']
==> Table description: This table likely represents restaurant listings including ratings, contact information, and addresses.
=> Schema description: This table likely represents restaurant listings including ratings, contact information, and addresses.
=> Overall schema: Index(['ID', 'NAME', 'RATING', 'PHONENUMBER', 'NO_OF_REVIEWS', 'ADDRESS'], dtype='object')
==> Reasoning: Considering the schema and the description, the `ID` column seems to be a unique identifier for each restaurant entry. Given that this table is about restaurant listings, a more descriptive name for thi

100%|██████████| 1/1 [01:53<00:00, 113.81s/it]

==> Reasoning: Considering the context of the table, which contains information about restaurants, the column "ADDRESS" can be renamed to something more specific that reflects its content. Given that the table includes details like city and state, a more descriptive name could be "Full Address".

New column name: Full Address
==> Renaming column ADDRESS to Full Address


[['RESTAURANT_ID',
  'RESTAURANT_NAME',
  'Customer_Rating',
  'TELEPHONE_NUMBER',
  'NUMBER_OF_REVIEWS',
  'Full Address']]

In [32]:
df = pd.read_csv('../../../data_src/yelp.csv')
get_enhanced_schema([df])

  0%|          | 0/1 [00:00<?, ?it/s]

==> Sample descriptions: ["This table likely represents restaurant information, including the restaurant's name, rating, phone number, number of reviews, and address.", 'The table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.', 'This table likely represents information about businesses, including their name, rating, phone number, number of reviews, and address.']
==> Table description: This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.
=> Schema description: This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.
=> Overall schema: Index(['ID', 'NAME', 'RATING', 'PHONENUMBER', 'NO_OF_REVIEWS', 'ADDRESS'], dtype='object')
==> Reasoning: Considering the context of the table, which appears to contain information abou

100%|██████████| 1/1 [02:08<00:00, 128.34s/it]

==> Reasoning: Considering the context of the table, which appears to contain information about restaurants or cafes, and looking at the existing columns such as ID, NAME, RATING, PHONENUMBER, NO_OF_REVIEWS, and ADDRESS, it would be beneficial to rename the ADDRESS column to something that clearly indicates its purpose within the dataset. A suitable new name could be "LOCATION".

New column name: LOCATION
==> Renaming column ADDRESS to LOCATION


[['RESTAURANT_ID',
  'RESTAURANT_NAME',
  'RESTAURANT_RATING',
  'PHONE_NUMBER_FORMATTED',
  'REVIEWS',
  'LOCATION']]

In [33]:
df = pd.read_csv('../../../data_src/table_73.csv')
get_enhanced_schema([df])

  0%|          | 0/1 [00:00<?, ?it/s]

==> Sample descriptions: ['The table likely represents wind energy data from different wind turbines (WTGs) for a specific time period. The columns suggest it includes turbine identifiers, timestamps, possibly wind speed or power measurements, and other metrics like temperatures or pressures. The last column "WNET (bin)" could be a binary classification indicating whether a certain condition was met (e.g., network status).', 'The table likely represents weather or environmental data, possibly from a specific location or station. The columns suggest it includes identifiers (F1, F2), timestamps (F3), temperature or pressure readings (F4, F5), and possibly other meteorological measurements (F6-F9). The last two columns (Number of Records and WNET (bin)) might indicate the number of records associated with each entry and a binary classification related to weather conditions, respectively.', "The table likely represents weather or environmental data collected from different sensors (indicat

100%|██████████| 1/1 [04:38<00:00, 278.77s/it]

==> Reasoning: After considering the schema, sample rows, and the description of the table, the column "WNET (bin)" seems to represent a binary classification related to a specific weather or environmental condition. To make the column more descriptive, we can consider renaming it to something that reflects its purpose more clearly.

Given that the column is a binary classification, a suitable name could be "Weather Condition Binary" or simply "Condition Binary". However, since the original name includes "WNET", which might refer to a specific network or system, we can keep that part to maintain context.

New column name: Weather Condition Binary (WNET)
==> Renaming column WNET (bin) to Weather Condition Binary (WNET)


[['SensorID',
  'SensorID',
  'Timestamp',
  'Pressure',
  'Temperature',
  'MissingValueFlag',
  'Temperature_Celsius',
  'RepetitionOfF9',
  'Pressure_Repeated_or_Confirmation',
  'Sensor Count',
  'Weather Condition Binary (WNET)']]

In [34]:
df = pd.read_csv('../../../data_src/table_5.csv')
get_enhanced_schema([df])

  0%|          | 0/1 [00:00<?, ?it/s]

==> Sample descriptions: ["This table appears to represent batting statistics for individual players in baseball, including various offensive metrics, ball flight data, and game context information. The rows contain detailed statistics such as batting average (AVG), on-base percentage (OBP), slugging percentage (SLG), and specific events like hits (H), home runs (HR), and walks (BB). It also includes information about the player's name, team, and league, as well as calculations related to their performance.", "This table appears to represent detailed statistics for baseball players, focusing on their performance in a specific season (2011). It includes various offensive and defensive metrics such as batting average, on-base percentage, slugging percentage, and more. The table also contains information about the player's name, team, and league.", 'This table appears to represent detailed statistics for baseball players, specifically focusing on pitching performance. The columns include 

100%|██████████| 1/1 [18:02<00:00, 1082.16s/it]

==> Reasoning: Considering the context of the table and the other columns, the column "Calculation_40532458115264531" seems to represent a calculated value or statistic. Given that it appears in multiple rows and likely corresponds to a specific metric, a more descriptive name could be "Weighted_On_Base_Average" (wOBA) since it aligns with the existing wOBA column and the nature of the calculation.

New column name: Weighted_On_Base_Average
==> Renaming column Calculation_40532458115264531 to Weighted_On_Base_Average


[['At_Bats',
  'Batting_Average',
  'Batting_Average_On_Balls_In_Play:\n\nThis new column name is more descriptive and aligns well with the existing column names that represent different types of batting statistics. It clearly indicates that the column contains the batting average on balls in play, which is a key offensive metric in baseball.',
  'Walks Percentage',
  'Walks',
  'BIP_Percentage:\n\nThis new column name is more descriptive and indicates that the values represent a percentage of batted balls that are classified as being in the air. This aligns well with the existing column names and provides clarity about the statistical metric being measured.',
  'BIP_%',
  'Pitching_Balls',
  'Called_Strikes',
  'Plate_Appears',
  'CalledStrike',
  'Foul_Balls',
  'Fly_Balls_Pct',
  'Fly_Balls',
  'Foul_Pitches:',
  'Ground_Ball_Rate',
  'GB_Pct',
  'Ground Into Double Play: \n\nThis name is more descriptive and explicit, clearly indicating that the column represents the number of time